In [2]:
import os
os.chdir('../../../../..')

In [3]:
import numpy as np
import hdbscan
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
import gc
import numpy as np
import polars as pl
from dscribe.descriptors import SOAP
from rdkit import Chem
from tqdm import tqdm
import numpy as np
import polars as pl
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np
import polars as pl
from ase import Atoms
from dscribe.descriptors import SOAP
from rdkit import Chem
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
import numpy as np
import polars as pl
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler





from sklearn.cluster import AgglomerativeClustering, SpectralClustering, DBSCAN
from kmedoids import KMedoids
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform
from umap import UMAP
from dscribe.kernels import REMatchKernel

from src.datasets import QM9Dataset
from src.helper_functions import plot_molecules_with_py3dmol, create_chemiscope_viewer, plot_distance_matrix_projection, evaluate_distance_matrix_clustering_sweep, average_numeric_by_cluster, evaluate_hdbscan_grid, get_isomers

projection_method = "MDS"

In [4]:
qm9 = QM9Dataset(limit=20_000, descriptors=["mace", "soap"])
df = qm9.load()

2026-06-11 10:53:50.626 | INFO     | src.datasets:_load_full_qm9_df:934 - Loading cached full QM9 dataset from: data/QM9/dataset_cleaned.parquet
2026-06-11 10:53:51.164 | INFO     | src.datasets:_sample_qm9_df:1118 - QM9 sampling complete: strategy=stratified, requested_limit=20000, returned_rows=20000, sampling on columns=['num_atoms', 'gap'].
2026-06-11 10:53:51.165 | INFO     | src.datasets:_add_requested_descriptors:292 - Applying requested QM9 descriptors to sampled dataframe (rows=20000).
2026-06-11 10:53:51.166 | INFO     | src.features:compute_mace_outputs:679 - Computing MACE embeddings (model=medium, batch_size=32)...
/Users/karlfindhansen/thesis/Anomaly-Detection-in-Molecular-and-Materials-Datasets/.venv/lib/python3.12/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.loa

cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.
Using MACE-OFF23 MODEL for MACECalculator with /Users/karlfindhansen/.cache/mace/MACE-OFF23_medium.model
Using float32 for MACECalculator, which is faster but less accurate. Recommended for MD. Use float64 for geometry optimization.


2026-06-11 11:25:04.921 | SUCCESS  | src.datasets:add_mace:1349 - Added MACE embeddings and matrices.
2026-06-11 11:25:04.956 | INFO     | src.features:compute_soap_outputs:395 - Computing SOAP (rcut=6.0, nmax=8, lmax=6, normalize=True)...
2026-06-11 11:25:32.966 | SUCCESS  | src.datasets:add_soap:1306 - Added SOAP embeddings and matrices.
2026-06-11 11:25:32.970 | INFO     | src.datasets:_add_requested_descriptors:303 - Added descriptor column(s): ['mace_embedding', 'mace_matrix', 'soap_embedding', 'soap_matrix']
2026-06-11 11:25:33.006 | INFO     | src.datasets:_load_with_descriptor_filter:974 - QM9 descriptor null-filtering complete: attempts=1, requested_limit=20000, returned_rows=20000, base_rows=20000.


# Hypothesis 1
- mace will be better to predict the dipole moment of isomers. because it is trained to understand the higher order geometries of a molecule. MACE uses equivariant message passing. In each layer, atoms pass directional, geometric information to their neighbors. Over multiple layers, this allows MACE to build a high-body-order, global topological map of the molecule. It natively tracks how a change on one side of the molecule affects the electronic asymmetry on the other side, allowing it to map the shifting dipole moments precisely.

In [8]:

# -------------------------------------------------------------------------
# 1. ISOLATE ACTUAL TOP ISOMER GROUP
# -------------------------------------------------------------------------
def isolate_top_isomers(
    df: pl.DataFrame,
    formula_col: str = "formula",
    min_isomers: int = 30,
) -> pl.DataFrame:
    formula_counts = (
        df.group_by(formula_col)
        .agg(pl.len().alias("count"))
        .filter(pl.col("count") >= min_isomers)
        .sort("count", descending=True)
    )
    if formula_counts.is_empty():
        raise ValueError(f"No formulas found with at least {min_isomers} isomers.")
    top_formula = formula_counts.item(0, formula_col)
    top_count = formula_counts.item(0, "count")
    print(f"[+] Isolating TRUE isomers for top formula: {top_formula} ({top_count} structures)")
    return df.filter(pl.col(formula_col) == top_formula)


# -------------------------------------------------------------------------
# 2. FEATURE EXTRACTION HELPERS
# -------------------------------------------------------------------------
def extract_averaged_embedding(df: pl.DataFrame, col: str) -> np.ndarray:
    """
    Extracts a pre-averaged molecular embedding (one vector per molecule).
    Used for the '*_embedding' columns where pooling has already been applied.
    Returns shape [n_molecules, D].
    """
    X = np.vstack(df[col].to_list())
    assert X.ndim == 2, (
        f"Expected 2D array from column '{col}', got shape {X.shape}. "
        f"If this column contains per-atom matrices, use extract_pooled_matrix() instead."
    )
    return X


def extract_pooled_matrix(df: pl.DataFrame, col: str) -> np.ndarray:
    """
    Extracts per-atom feature matrices and reduces each molecule to a single
    vector via mean pooling across atoms.
    Used for the '*_matrix' columns where pooling has not yet been applied.
    Returns shape [n_molecules, D].
    """
    pooled = []
    for mat in df[col].to_list():
        arr = np.array(mat)
        if arr.ndim == 1:
            pooled.append(arr)
        else:
            pooled.append(arr.mean(axis=0))
    return np.vstack(pooled)


def extract_features(df: pl.DataFrame, col: str) -> np.ndarray:
    """
    Routes to the correct extraction strategy based on column naming convention:
      - '*_embedding' columns are pre-averaged, extracted directly.
      - '*_matrix' columns contain per-atom arrays, pooled via mean before use.
    """
    if col.endswith("_embedding"):
        return extract_averaged_embedding(df, col)
    elif col.endswith("_matrix"):
        return extract_pooled_matrix(df, col)
    else:
        raise ValueError(
            f"Column '{col}' does not follow the naming convention. "
            f"Expected a column ending in '_embedding' (pre-averaged) "
            f"or '_matrix' (per-atom, will be mean-pooled)."
        )


def sanity_check(
    df: pl.DataFrame,
    soap_col: str,
    mace_col: str,
    property_col: str,
) -> None:
    """
    Verifies that all arrays share the same number of samples before any
    fitting is attempted. Prints shapes for manual inspection.
    """
    X_soap = extract_features(df, soap_col)
    X_mace = extract_features(df, mace_col)
    y = df[property_col].to_numpy()

    print(f"[Sanity Check]")
    print(f"  SOAP  '{soap_col}' shape : {X_soap.shape}")
    print(f"  MACE  '{mace_col}' shape : {X_mace.shape}")
    print(f"  y     '{property_col}' shape : {y.shape}")

    assert X_soap.shape[0] == y.shape[0], (
        f"SOAP/y mismatch: {X_soap.shape[0]} vs {y.shape[0]}"
    )
    assert X_mace.shape[0] == y.shape[0], (
        f"MACE/y mismatch: {X_mace.shape[0]} vs {y.shape[0]}"
    )
    print("  [OK] All sample dimensions consistent.\n")


# -------------------------------------------------------------------------
# 3. SHARED NESTED CV — RBF KRR
# -------------------------------------------------------------------------
def evaluate_rbf_krr(
    X: np.ndarray,
    y: np.ndarray,
    n_splits: int = 5,
    label: str = "Model",
) -> tuple[float, float, float]:
    """
    Evaluates any fixed-length feature matrix using nested cross-validation
    over an expansive RBF KRR hyperparameter grid spanning 9 orders of magnitude.

    Using the same non-linear regressor for both SOAP and MACE ensures that
    any performance difference reflects representation quality rather than
    regressor choice.

    Returns: (mean_mae, std_mae, mean_r2)
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    maes, r2s = [], []

    param_grid = {
        "krr__alpha": np.logspace(-6, 2, 9).tolist(),
        "krr__gamma": np.logspace(-8, 0, 9).tolist(),
    }

    print(f" -> Beginning Nested CV [{label}] with RBF KRR. "
          f"Grid size: {9 * 9} combinations.")

    for train_idx, test_idx in tqdm(
        kf.split(X, y), desc=f"Outer Loops ({label})", total=n_splits
    ):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("krr", KernelRidge(kernel="rbf")),
        ])
        grid_search = GridSearchCV(
            estimator=pipe,
            param_grid=param_grid,
            cv=3,
            scoring="neg_mean_absolute_error",
            n_jobs=-1,
        )
        grid_search.fit(X_train, y_train)
        preds = grid_search.predict(X_test)
        maes.append(mean_absolute_error(y_test, preds))
        r2s.append(r2_score(y_test, preds))

    return float(np.mean(maes)), float(np.std(maes)), float(np.mean(r2s))


# -------------------------------------------------------------------------
# 4. SHARED NESTED CV — Linear Ridge
# -------------------------------------------------------------------------
def evaluate_linear_ridge(
    X: np.ndarray,
    y: np.ndarray,
    n_splits: int = 5,
    label: str = "Model",
) -> tuple[float, float, float]:
    """
    Evaluates any fixed-length feature matrix using a linear Ridge probe
    with built-in cross-validated penalty selection.

    Running this on both SOAP and MACE tests whether MACE's advantage
    persists under a simpler regressor, or whether SOAP can match it
    when freed from kernel saturation concerns in high-dimensional spaces.

    Returns: (mean_mae, std_mae, mean_r2)
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    maes, r2s = [], []

    wide_alphas = np.logspace(-6, 6, 13)

    print(f" -> Beginning Nested CV [{label}] with Linear RidgeCV.")

    for train_idx, test_idx in tqdm(
        kf.split(X, y), desc=f"Outer Loops ({label})", total=n_splits
    ):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("ridge", RidgeCV(alphas=wide_alphas, cv=3)),
        ])
        pipe.fit(X_train, y_train)
        preds = pipe.predict(X_test)
        maes.append(mean_absolute_error(y_test, preds))
        r2s.append(r2_score(y_test, preds))

    return float(np.mean(maes)), float(np.std(maes)), float(np.mean(r2s))


# -------------------------------------------------------------------------
# 5. CONVENIENCE WRAPPERS
# -------------------------------------------------------------------------
def evaluate_soap_rbf_krr(
    df: pl.DataFrame,
    soap_col: str,
    property_col: str,
    n_splits: int = 5,
) -> tuple[float, float, float]:
    """SOAP evaluated with the non-linear RBF KRR regressor."""
    X = extract_features(df, soap_col)
    y = df[property_col].to_numpy()
    return evaluate_rbf_krr(X, y, n_splits=n_splits, label=f"{soap_col} / RBF-KRR")


def evaluate_soap_linear_ridge(
    df: pl.DataFrame,
    soap_col: str,
    property_col: str,
    n_splits: int = 5,
) -> tuple[float, float, float]:
    """
    SOAP evaluated with a linear Ridge probe.
    High-dimensional SOAP vectors are not expected to be linearly separable,
    so this serves as a lower-bound reference rather than a primary result.
    """
    X = extract_features(df, soap_col)
    y = df[property_col].to_numpy()
    return evaluate_linear_ridge(X, y, n_splits=n_splits, label=f"{soap_col} / Linear-Ridge")


def evaluate_mace_rbf_krr(
    df: pl.DataFrame,
    mace_col: str,
    property_col: str,
    n_splits: int = 5,
) -> tuple[float, float, float]:
    """
    MACE evaluated with the same non-linear RBF KRR regressor used for SOAP.
    This is the symmetry-preserving comparison: if MACE still outperforms SOAP
    under an identical non-linear readout, the advantage is attributable to the
    representation rather than regressor choice.
    """
    X = extract_features(df, mace_col)
    y = df[property_col].to_numpy()
    return evaluate_rbf_krr(X, y, n_splits=n_splits, label=f"{mace_col} / RBF-KRR")


def evaluate_mace_linear_ridge(
    df: pl.DataFrame,
    mace_col: str,
    property_col: str,
    n_splits: int = 5,
) -> tuple[float, float, float]:
    """
    MACE evaluated with a linear Ridge probe.
    Because MACE latent vectors encode geometric information in a roughly
    linearized space, this is the natural readout for MACE and serves as
    its primary result.
    """
    X = extract_features(df, mace_col)
    y = df[property_col].to_numpy()
    return evaluate_linear_ridge(X, y, n_splits=n_splits, label=f"{mace_col} / Linear-Ridge")


# -------------------------------------------------------------------------
# 6. UNIFIED RUNNER — produces the full 2x2 comparison table
# -------------------------------------------------------------------------
def run_symmetric_comparison(
    df: pl.DataFrame,
    soap_col: str,
    mace_col: str,
    property_col: str,
    n_splits: int = 5,
) -> pl.DataFrame:
    """
    Runs all four regressor/representation combinations and returns a tidy
    DataFrame suitable for direct inclusion as a results table.

    The 2x2 design is:
                        RBF KRR (non-linear)    Linear Ridge
        SOAP            [primary]               [lower bound]
        MACE            [symmetry check]        [natural readout]

    Accepted column naming conventions:
      - soap_col / mace_col ending in '_embedding': pre-averaged, used directly.
      - soap_col / mace_col ending in '_matrix': per-atom arrays, mean-pooled here.

    If MACE / RBF-KRR still outperforms SOAP / RBF-KRR, the advantage is
    representation-driven. If the gap shrinks or reverses, the original
    MACE / Linear-Ridge result was partly a regressor artifact.
    """
    sanity_check(df, soap_col, mace_col, property_col)

    configs = [
        ("SOAP", "RBF KRR",      evaluate_soap_rbf_krr,      soap_col),
        ("SOAP", "Linear Ridge", evaluate_soap_linear_ridge, soap_col),
        ("MACE", "RBF KRR",      evaluate_mace_rbf_krr,      mace_col),
        ("MACE", "Linear Ridge", evaluate_mace_linear_ridge, mace_col),
    ]

    results = []
    for representation, regressor, fn, col in configs:
        print(f"\n{'='*60}")
        print(f"  {representation} + {regressor}  (column: '{col}')")
        print(f"{'='*60}")
        mae_mean, mae_std, r2_mean = fn(df, col, property_col, n_splits)
        results.append({
            "Representation": representation,
            "Column":         col,
            "Regressor":      regressor,
            "Mean MAE":       round(mae_mean, 4),
            "Std MAE":        round(mae_std,  4),
            "Mean R²":        round(r2_mean,  4),
        })
        print(f"  MAE: {mae_mean:.4f} ± {mae_std:.4f}   R²: {r2_mean:.4f}")

    return pl.DataFrame(results)

In [6]:
# NOTE: must try to run this with a much larger subsample of isomers (should have at least )
isomer_subset = isolate_top_isomers(df, formula_col="formula", min_isomers=30)
sanity_check(isomer_subset, "soap_embedding", "mace_embedding", "mu")



results = run_symmetric_comparison(
    isomer_subset,
    soap_col="soap_embedding",
    mace_col="mace_embedding",
    property_col="mu",
)
print(results)

results_matrix = run_symmetric_comparison(
    isomer_subset,
    soap_col="soap_matrix",
    mace_col="mace_matrix",
    property_col="mu",
)
print(results_matrix)

[+] Isolating TRUE isomers for top formula: C7H10O2 (890 structures)
[Sanity Check]
  SOAP  'soap_embedding' shape : (890, 252)
  MACE  'mace_embedding' shape : (890, 256)
  y     'mu' shape : (890,)
  [OK] All sample dimensions consistent.

[Sanity Check]
  SOAP  'soap_embedding' shape : (890, 252)
  MACE  'mace_embedding' shape : (890, 256)
  y     'mu' shape : (890,)
  [OK] All sample dimensions consistent.


  SOAP + RBF KRR  (column: 'soap_embedding')
 -> Beginning Nested CV [soap_embedding / RBF-KRR] with RBF KRR. Grid size: 81 combinations.


Outer Loops (soap_embedding / RBF-KRR): 100%|██████████| 5/5 [00:18<00:00,  3.61s/it]


  MAE: 0.8066 ± 0.0410   R²: 0.0527

  SOAP + Linear Ridge  (column: 'soap_embedding')
 -> Beginning Nested CV [soap_embedding / Linear-Ridge] with Linear RidgeCV.


Outer Loops (soap_embedding / Linear-Ridge): 100%|██████████| 5/5 [00:00<00:00, 14.58it/s]


  MAE: 0.8098 ± 0.0426   R²: 0.0558

  MACE + RBF KRR  (column: 'mace_embedding')
 -> Beginning Nested CV [mace_embedding / RBF-KRR] with RBF KRR. Grid size: 81 combinations.


...

  MAE: 0.6783 ± 0.0206   R²: 0.3197

  MACE + RBF KRR  (column: 'mace_matrix')
 -> Beginning Nested CV [mace_matrix / RBF-KRR] with RBF KRR. Grid size: 81 combinations.


Outer Loops (mace_matrix / RBF-KRR): 100%|██████████| 5/5 [00:04<00:00,  1.05it/s]


  MAE: 0.4690 ± 0.0195   R²: 0.6384

  MACE + Linear Ridge  (column: 'mace_matrix')
 -> Beginning Nested CV [mace_matrix / Linear-Ridge] with Linear RidgeCV.


Outer Loops (mace_matrix / Linear-Ridge): 100%|██████████| 5/5 [00:00<00:00, 10.31it/s]

  MAE: 0.4912 ± 0.0228   R²: 0.6133
shape: (4, 6)
┌────────────────┬─────────────┬──────────────┬──────────┬─────────┬─────────┐
│ Representation ┆ Column      ┆ Regressor    ┆ Mean MAE ┆ Std MAE ┆ Mean R² │
│ ---            ┆ ---         ┆ ---          ┆ ---      ┆ ---     ┆ ---     │
│ str            ┆ str         ┆ str          ┆ f64      ┆ f64     ┆ f64     │
╞════════════════╪═════════════╪══════════════╪══════════╪═════════╪═════════╡
│ SOAP           ┆ soap_matrix ┆ RBF KRR      ┆ 0.6785   ┆ 0.0199  ┆ 0.3104  │
│ SOAP           ┆ soap_matrix ┆ Linear Ridge ┆ 0.6783   ┆ 0.0206  ┆ 0.3197  │
│ MACE           ┆ mace_matrix ┆ RBF KRR      ┆ 0.469    ┆ 0.0195  ┆ 0.6384  │
│ MACE           ┆ mace_matrix ┆ Linear Ridge ┆ 0.4912   ┆ 0.0228  ┆ 0.6133  │
└────────────────┴─────────────┴──────────────┴──────────┴─────────┴─────────┘


In [7]:
df

mol_id,formula,smiles,canonical_smiles,scaffold_smiles,generic_scaffold,root_scaffold,brics_fragments,scaffold_tree_nodes,selfies,functional_groups,structure_class,is_injected,outlier_category,mol_weight,logp,tpsa,election_affinity,ionization_energies,num_heavy_atoms,num_rings,num_aromatic_rings,num_fluorine,num_heteroatoms,num_atoms,coordination,num_rotatable_bonds,fraction_csp1,fraction_csp2,fraction_csp3,h_bond_donors,h_bond_acceptors,branching_index,num_sp_carbons,num_sp2_carbons,num_sp3_carbons,main_chain_length,…,fr_benzene,fr_alcohol,fr_phenol,fr_amine,fr_amide,fr_carboxylic_acid,fr_ester,fr_ketone,fr_ether,fr_nitro,pbf_score,coordinates,atomic_numbers,mu,alpha,homo,lumo,gap,r2,zpve,u0,u,h,g,cv,u0_atom,u_atom,h_atom,g_atom,A,B,C,geometric_strain,mace_embedding,mace_matrix,soap_embedding,soap_matrix
str,str,str,str,str,str,str,str,str,str,str,str,i64,str,i64,i64,i64,f64,f64,i64,i64,i64,i64,i64,i64,f64,i64,f64,f64,f64,i64,i64,i64,i64,i64,i64,i64,…,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,list[list[f64]],list[i64],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,list[f64],list[list[f64]],list[f64],list[list[f64]]
"""qm9_2""","""H2O""","""[H]O[H]""","""O""","""Acyclic""","""Acyclic""","""Acyclic""","""[H]O[H]""","""""","""[O]""","""""","""Acyclic""",0,null,18,0,31,0.989835,13.604975,1,0,0,0,1,3,1.333333,0,0.0,0.0,0.0,0,0,0,0,0,0,2,…,0,0,0,0,0,0,0,0,0,0,0.0,"[[-0.0344, 0.9775, 0.0076], [0.0648, 0.0206, 0.0015], [0.8718, 1.3008, 0.0007]]","[8, 1, 1]",1.8511,6.31,-7.967494,1.869422,9.836916,19.0002,0.581643,-2079.077881,-2079.000732,-2078.975098,-2079.558105,6.002,-9.240362,-9.278811,-9.330214,-8.733849,799.588135,437.90387,282.945465,33.146867,"[0.151596, -0.116375, … 0.065697]","[[0.272173, -0.260508, … 0.135482], [0.091303, -0.044316, … 0.030799], [0.091312, -0.0443, … 0.030809]]","[0.014712, 0.043991, … 0.00007]","[[0.011366, 0.037117, … 0.000003], [0.016255, 0.046418, … 0.000183], [0.016255, 0.046415, … 0.000183]]"
"""qm9_3""","""C2H2""","""[H]C#C[H]""","""C#C""","""Acyclic""","""Acyclic""","""Acyclic""","""[H]C#C[H]""","""""","""[C][#C]""","""""","""Acyclic""",0,null,26,0,0,1.008157,12.429361,2,0,0,0,0,4,1.5,0,1.0,0.0,0.0,0,0,0,2,0,0,3,…,0,0,0,0,0,0,0,0,0,0,0.0,"[[0.5995, 0.0, 1.0], [-0.5995, 0.0, 1.0], … [1.6616, 0.0, 1.0]]","[6, 6, … 1]",0.0,16.280001,-7.741639,1.376896,9.118535,59.524799,0.730381,-2103.669434,-2103.590576,-2103.564697,-2104.186523,8.574,-16.716963,-16.792231,-16.869347,-15.862634,0.0,35.610035,35.610035,0.0,"[-0.070554, 0.186363, … 0.112174]","[[-0.192572, 0.362839, … 0.194028], [-0.192572, 0.362839, … 0.194028], … [0.051464, 0.009887, … 0.03032]]","[0.016141, 0.046618, … 0.01816]","[[0.012223, 0.035042, … 0.004926], [0.012223, 0.035042, … 0.004926], … [0.020414, 0.05937, … 0.043569]]"
"""qm9_5""","""CH2O""","""[H]C([H])=O""","""C=O""","""Acyclic""","""Acyclic""","""Acyclic""","""[H]C([H])=O""","""""","""[C][=O]""","""""","""Acyclic""",0,null,30,0,17,1.057906,13.018803,2,0,0,0,1,4,1.5,0,0.0,1.0,0.0,0,1,1,0,1,0,2,…,0,0,0,0,0,0,0,0,0,0,0.000004,"[[-0.014, 1.1802, 0.0078], [0.0023, -0.0197, 0.0022], … [-0.9591, 1.764, 0.0172]]","[6, 8, … 1]",2.1089,14.18,-7.26544,-1.104782,6.157937,59.989101,0.723904,-3115.257812,-3115.179688,-3115.154053,-3115.846924,6.413,-15.557186,-15.633324,-15.710442,-14.763947,285.488403,38.9823,34.29892,12.315466,"[-0.02423, 0.003419, … 0.045106]","[[-0.225969, 0.112141, … 0.0902], [0.069587, -0.110025, … 0.026074], … [0.029734, 0.005779, … 0.032075]]","[0.015004, 0.042045, … 0.002088]","[[0.00761, 0.02144, … 0.000035], [0.018358, 0.050692, … 0.006216], … [0.016958, 0.047757, … 0.004815]]"
"""qm9_9""","""C2H3N""","""[H]C([H])([H])C#N""","""CC#N""","""Acyclic""","""Acyclic""","""Acyclic""","""[H]C([H])([H])C#N""","""""","""[C][C][#N]""","""""","""Acyclic""",0,null,41,0,23,0.5644705,12.975002,3,0,0,0,1,6,1.666667,0,0.5,0.0,0.5,0,1,1,1,0,1,3,…,0,0,0,0,0,0,0,0,0,0,0.339715,"[[-0.0179, 1.4671, 0.0101], [0.0017, 0.0104, 0.0021], … [-0.5271, 1.8352, 0.9

In [9]:
results = run_symmetric_comparison(
    isomer_subset,
    soap_col="soap_embedding",
    mace_col="mace_embedding",
    property_col="gap",
)
print(results)

results_matrix = run_symmetric_comparison(
    isomer_subset,
    soap_col="soap_matrix",
    mace_col="mace_matrix",
    property_col="gap",
)
print(results_matrix)

[Sanity Check]
  SOAP  'soap_embedding' shape : (890, 252)
  MACE  'mace_embedding' shape : (890, 256)
  y     'gap' shape : (890,)
  [OK] All sample dimensions consistent.


  SOAP + RBF KRR  (column: 'soap_embedding')
 -> Beginning Nested CV [soap_embedding / RBF-KRR] with RBF KRR. Grid size: 81 combinations.


Outer Loops (soap_embedding / RBF-KRR): 100%|██████████| 5/5 [00:03<00:00,  1.53it/s]


  MAE: 0.6727 ± 0.0558   R²: 0.4835

  SOAP + Linear Ridge  (column: 'soap_embedding')
 -> Beginning Nested CV [soap_embedding / Linear-Ridge] with Linear RidgeCV.


Outer Loops (soap_embedding / Linear-Ridge): 100%|██████████| 5/5 [00:00<00:00, 11.46it/s]


  MAE: 0.6752 ± 0.0531   R²: 0.4798

  MACE + RBF KRR  (column: 'mace_embedding')
 -> Beginning Nested CV [mace_embedding / RBF-KRR] with RBF KRR. Grid size: 81 combinations.


...

  MAE: 0.4134 ± 0.0175   R²: 0.8029

  MACE + RBF KRR  (column: 'mace_matrix')
 -> Beginning Nested CV [mace_matrix / RBF-KRR] with RBF KRR. Grid size: 81 combinations.


Outer Loops (mace_matrix / RBF-KRR): 100%|██████████| 5/5 [00:03<00:00,  1.40it/s]


  MAE: 0.1505 ± 0.0062   R²: 0.9706

  MACE + Linear Ridge  (column: 'mace_matrix')
 -> Beginning Nested CV [mace_matrix / Linear-Ridge] with Linear RidgeCV.


Outer Loops (mace_matrix / Linear-Ridge): 100%|██████████| 5/5 [00:00<00:00, 15.31it/s]

  MAE: 0.3003 ± 0.0185   R²: 0.8905
shape: (4, 6)
┌────────────────┬─────────────┬──────────────┬──────────┬─────────┬─────────┐
│ Representation ┆ Column      ┆ Regressor    ┆ Mean MAE ┆ Std MAE ┆ Mean R² │
│ ---            ┆ ---         ┆ ---          ┆ ---      ┆ ---     ┆ ---     │
│ str            ┆ str         ┆ str          ┆ f64      ┆ f64     ┆ f64     │
╞════════════════╪═════════════╪══════════════╪══════════╪═════════╪═════════╡
│ SOAP           ┆ soap_matrix ┆ RBF KRR      ┆ 0.3579   ┆ 0.0093  ┆ 0.8498  │
│ SOAP           ┆ soap_matrix ┆ Linear Ridge ┆ 0.4134   ┆ 0.0175  ┆ 0.8029  │
│ MACE           ┆ mace_matrix ┆ RBF KRR      ┆ 0.1505   ┆ 0.0062  ┆ 0.9706  │
│ MACE           ┆ mace_matrix ┆ Linear Ridge ┆ 0.3003   ┆ 0.0185  ┆ 0.8905  │
└────────────────┴─────────────┴──────────────┴──────────┴─────────┴─────────┘


In [27]:
isomer_subset = isolate_top_isomers(df, formula_col="formula", min_isomers=30)

print(
    f"Evaluating dipole regression across {isomer_subset.height} closely matched isomers...\n"
)

soap_mae, soap_std, soap_r2 = evaluate_soap_rematch(
    isomer_subset, "soap_embedding", "mu"
)

print(
    f"SOAP (RBF Kernel CV) -> Dipole MAE: {soap_mae:.4f} ± {soap_std:.4f} | R² Score: {soap_r2:.4f}"
)

mace_mae, mace_std, mace_r2 = evaluate_linear_mace(
    isomer_subset, "mace_embedding", "mu"
)

print(
    f"MACE (Linear RidgeCV) -> Dipole MAE: {mace_mae:.4f} ± {mace_std:.4f} | R² Score: {mace_r2:.4f}"
)

[+] Isolating TRUE isomers for top formula: C7H10O2 (890 structures)
Evaluating dipole regression across 890 closely matched isomers...

 -> Beginning Nested CV for SOAP. Inner search grid size: 81 combinations.


Outer Loops (SOAP): 100%|██████████| 5/5 [00:09<00:00,  1.85s/it]


SOAP (RBF Kernel CV) -> Dipole MAE: 0.8066 ± 0.0410 | R² Score: 0.0527
MACE (Linear RidgeCV) -> Dipole MAE: 0.4912 ± 0.0228 | R² Score: 0.6133


# Hypothesis 2

In [ ]:
def run_linear_probe(df, embedding_col, target_col):
    X = np.vstack(df[embedding_col].to_numpy())
    y = df[target_col].to_numpy()
    
    # Filter out classes with very few samples to ensure clean cross-validation
    unique, counts = np.unique(y, return_counts=True)
    valid_classes = unique[counts >= 10]
    
    # Filter matrix and labels
    mask = np.isin(y, valid_classes)
    X, y = X[mask], y[mask]
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    accuracies = []
    
    for train_idx, test_idx in skf.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        # Linear probe (No hidden layers, purely tests linear separability)
        model = LogisticRegression(max_iter=1000, random_state=42)
        model.fit(X_train_scaled, y_train)
        
        preds = model.predict(X_test_scaled)
        accuracies.append(accuracy_score(y_test, preds))
        
    return np.mean(accuracies), np.std(accuracies)

# --- Evaluate both spaces on "num_rings" ---
print("Evaluating Linear Separability of Topological Concepts...")

mace_acc, mace_std = run_linear_probe(df, "mace_embedding", "num_rings")
print(f"MACE -> Ring Classification Accuracy: {mace_acc:.4f} ± {mace_std:.4f}")

soap_acc, soap_std = run_linear_probe(df, "soap_embedding", "num_rings")
print(f"SOAP -> Ring Classification Accuracy: {soap_acc:.4f} ± {soap_std:.4f}")

Evaluating Linear Separability of Topological Concepts...
MACE -> Ring Classification Accuracy: 0.9851 ± 0.0016
SOAP -> Ring Classification Accuracy: 0.8399 ± 0.0063


# Hypothesis 2

In [ ]:
...

Ellipsis